# BBB Dataset — Exploratory Data Analysis

Interactive EDA for the curated BBB peptide dataset. Run after `bbb-dataset-build` (and optionally `bbb-dataset-augment`).

**Inputs:** `data/processed/peptides_bbb.parquet`, optional `peptides_bbb_with_augmentation.parquet`  
**Outputs:** tables and figures under `data/processed/eda_figures/`

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

BASE_DIR = Path("..").resolve()  # packages/dataset
PROCESSED = BASE_DIR / "data" / "processed"
EDA_DIR = PROCESSED / "eda_figures"

gold_df = pd.read_parquet(PROCESSED / "peptides_bbb.parquet")
combined_path = PROCESSED / "peptides_bbb_with_augmentation.parquet"
combined_df = pd.read_parquet(combined_path) if combined_path.exists() else gold_df.copy()
aug_df = combined_df[combined_df.get("is_augmented", 0) == 1].copy() if "is_augmented" in combined_df.columns else combined_df.iloc[0:0]

print(f"Gold rows: {len(gold_df)}")
print(f"Combined rows: {len(combined_df)}")
print(f"Augmented-only rows: {len(aug_df)}")

In [ ]:
FEATURE_COLS = [
    "mw", "hydrophobic_ratio_pct", "pi", "net_charge_ph7", "total_charge",
    "mean_hydrophobicity", "aliphatic_index", "boman_index", "aromaticity",
    "instability_index", "gravy",
]
COMPARISON_FEATURES = ["length", "gravy", "net_charge_ph7", "boman_index", "instability_index"]


def dataset_overview_table(df: pd.DataFrame, *, name: str = "dataset") -> pd.DataFrame:
    pos = int((df["bbb_label"] == 1).sum())
    neg = int((df["bbb_label"] == 0).sum())
    return pd.DataFrame([{
        "dataset": name,
        "rows": len(df),
        "bbb_positive": pos,
        "bbb_negative": neg,
        "positive_ratio": round(pos / len(df), 4) if len(df) else 0.0,
        "length_median": float(df["length"].median()) if len(df) else float("nan"),
        "unique_sequences": int(df["sequence"].nunique()),
        "unique_clusters": int(df["cluster_id"].nunique()) if "cluster_id" in df.columns else float("nan"),
    }])


def fold_overview_table(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for fold_id, group in df.groupby("fold_id", sort=True):
        rows.append({
            "fold_id": int(fold_id),
            "rows": len(group),
            "bbb_positive": int((group["bbb_label"] == 1).sum()),
            "positive_ratio": round(float(group["bbb_label"].mean()), 4),
            "unique_clusters": int(group["cluster_id"].nunique()),
            "external_test_rows": int(group.get("external_test", pd.Series([0] * len(group))).sum()),
        })
    return pd.DataFrame(rows).sort_values("fold_id").reset_index(drop=True)


def cluster_leakage_table(df: pd.DataFrame) -> pd.DataFrame:
    cv = df[(df.get("external_test", 0) == 0) & (df["fold_id"] >= 0)].copy()
    if cv.empty:
        return pd.DataFrame(columns=["cluster_id", "n_folds", "fold_ids"])
    grouped = cv.groupby("cluster_id")["fold_id"].agg(
        n_folds="nunique", fold_ids=lambda s: sorted(set(int(x) for x in s))
    ).reset_index()
    return grouped[grouped["n_folds"] > 1].sort_values("n_folds", ascending=False).reset_index(drop=True)


def save_fig(fig: plt.Figure, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)

In [ ]:
gold_out = EDA_DIR / "gold"
gold_out.mkdir(parents=True, exist_ok=True)

overview = dataset_overview_table(gold_df, name="gold")
folds = fold_overview_table(gold_df)
leakage = cluster_leakage_table(gold_df)
overview.to_csv(gold_out / "overview.csv", index=False)
folds.to_csv(gold_out / "fold_overview.csv", index=False)
leakage.to_csv(gold_out / "cluster_leakage.csv", index=False)
display(overview)
display(folds)
display(leakage if not leakage.empty else "Cluster leakage across folds: none detected.")

fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=gold_df, x="bbb_label", hue="split", ax=ax)
ax.set_title("BBB class balance by split")
save_fig(fig, gold_out / "class_balance.png")

fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(data=gold_df, x="length", hue="bbb_label", kde=True, bins=25, element="step", ax=ax)
ax.set_title("Length distribution")
save_fig(fig, gold_out / "length_distribution.png")

corr = gold_df[FEATURE_COLS].corr(method="spearman")
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, cmap="vlag", center=0, ax=ax)
ax.set_title("Spearman correlation (features)")
save_fig(fig, gold_out / "correlation_heatmap.png")

In [ ]:
if not aug_df.empty:
    aug_out = EDA_DIR / "augmentation"
    aug_out.mkdir(parents=True, exist_ok=True)

    overview_pp = pd.concat([
        dataset_overview_table(gold_df, name="gold"),
        dataset_overview_table(aug_df, name="augmented_extra"),
        dataset_overview_table(combined_df, name="gold_plus_augmented"),
    ], ignore_index=True)
    overview_pp.to_csv(aug_out / "overview_pre_post.csv", index=False)
    display(overview_pp)

    fig, ax = plt.subplots(figsize=(6, 4))
    overview_pp.set_index("dataset")["rows"].plot(kind="bar", ax=ax)
    ax.set_title("Dataset size before / after augmentation")
    save_fig(fig, aug_out / "row_counts_pre_post.png")

    fig, ax = plt.subplots(figsize=(7, 4))
    sns.kdeplot(data=gold_df, x="length", label="gold", fill=True, alpha=0.35, ax=ax)
    sns.kdeplot(data=aug_df, x="length", label="augmented", fill=True, alpha=0.35, ax=ax)
    ax.legend()
    ax.set_title("Length distribution: gold vs augmented")
    save_fig(fig, aug_out / "length_pre_post.png")
else:
    print("No augmented dataset found; run bbb-dataset-augment first for pre/post comparison.")